In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/Cancer Evolution Arena"
DATA_DIR = f"{PROJECT_DIR}/Data"
LUAD_DIR = f"{DATA_DIR}/luad_tcga_pub"

mutation_path = f"{LUAD_DIR}/data_mutations.txt"
clinical_path = f"{LUAD_DIR}/data_clinical_patient.txt"

print(os.path.exists(mutation_path))
print(os.path.exists(clinical_path))

Mounted at /content/drive
True
True


In [2]:
df = pd.read_csv(
    mutation_path,
    sep="\t",
    comment="#",
    low_memory=False
)

patient_gene = pd.crosstab(
    df["Tumor_Sample_Barcode"],
    df["Hugo_Symbol"]
)

patient_gene = (patient_gene > 0).astype(int)

patient_gene.index = [x[:12] for x in patient_gene.index]

print(patient_gene.shape)
patient_gene.head()

(230, 15130)


Hugo_Symbol,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1,ZZZ3,snoU13
TCGA-05-4249,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-05-4382,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-05-4384,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-05-4389,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-05-4390,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [3]:
cancer_genes_v2 = [
    "TP53","KRAS","EGFR","STK11","KEAP1",
    "NF1","RB1","MET","PIK3CA","BRAF",
    "ALK","RET","ROS1","ERBB4","ERBB2",
    "NTRK1","NTRK2","NTRK3","FGFR1","FGFR2",
    "FGFR3","FGFR4","ATM","ATR","ATRX",
    "CDKN2A","CCND1","MYC","PTEN","AKT1",
    "AKT2","AKT3","MAP2K1","MAP2K2","MAPK1",
    "NOTCH1","NOTCH2","NOTCH3","SMARCA4","SETD2",
    "RBM10","CTNNB1","DDR2","JAK1","STAT3",
    "STAT5B","RIT1","RAF1","TSC1","TSC2"
]

In [5]:
ecosystem_df = patient_gene[cancer_genes_v2].copy()

ecosystem_df.head()

Hugo_Symbol,TP53,KRAS,EGFR,STK11,KEAP1,NF1,RB1,MET,PIK3CA,BRAF,...,RBM10,CTNNB1,DDR2,JAK1,STAT3,STAT5B,RIT1,RAF1,TSC1,TSC2
TCGA-05-4249,0,1,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
TCGA-05-4382,1,0,1,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
TCGA-05-4384,1,0,0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,1,0,0,0
TCGA-05-4389,0,0,0,0,1,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0
TCGA-05-4390,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
from sklearn.cluster import AgglomerativeClustering

cluster_model = AgglomerativeClustering(
    n_clusters=4
)

clusters = cluster_model.fit_predict(
    ecosystem_df
)

ecosystem_df["Cluster"] = clusters

ecosystem_df["Cluster"].value_counts()

,count
Cluster,
2,81
0,66
1,57
3,26


In [7]:
cluster_summary = ecosystem_df.groupby(
    "Cluster"
).mean()

cluster_summary

Hugo_Symbol,TP53,KRAS,EGFR,STK11,KEAP1,NF1,RB1,MET,PIK3CA,BRAF,...,RBM10,CTNNB1,DDR2,JAK1,STAT3,STAT5B,RIT1,RAF1,TSC1,TSC2
Cluster,,,,,,,,,,,,,,,,,,,,,
0,0.212121,0.833333,0.000000,0.469697,0.212121,0.045455,0.030303,0.045455,0.000000,0.030303,...,0.196970,0.015152,0.030303,0.030303,0.015152,0.015152,0.000000,0.000000,0.015152,0.015152
1,0.140351,0.122807,0.035088,0.017544,0.350877,0.052632,0.000000,0.175439,0.017544,0.052632,...,0.035088,0.122807,0.017544,0.017544,0.017544,0.000000,0.070175,0.017544,0.000000,0.035088
2,0.876543,0.148148,0.086420,0.098765,0.061728,0.283951,0.074074,0.061728,0.172840,0.197531,...,0.049383,0.012346,0.086420,0.061728,0.024691,0.024691,0.012346,0.012346,0.049383,0.024691
3,0.538462,0.038462,1.000000,0.000000,0.038462,0.038462,0.076923,0.000000,0.076923,0.038462,...,0.038462,0.000000,0.000000,0.000000,0.038462,0.000000,0.000000,0.000000,0.000000,0.000000
